# LOS/NLOS labels da coordinate CSV/JSON (+ DEM)

Per ogni punto mette un ricevitore a terra, calcola i satelliti da RINEX NAV e classifica **LOS/NLOS**:
- edifici → BVH ray-trace
- terreno → DEM horizon (opzionale ma **obbligatorio per Genova**)

Script: `experiments/label_los_nlos_coords.py`

### Accortezze Genova (da pipeline area-map / rqz)
1. **`--rx-alt-mode dem`**: non usare alt ellissoidale fissa 1.5 m (colline).
2. **`--dem-auto-download`** (o `--dem-path`): horizon terrain; senza DEM le colline non occludono.
3. **`--geoid-correction egm96`**: tile AWS auto-DEM sono **ortometriche**. Senza EGM96, RX+edifici ~45–50 m troppo bassi → LOS/terrain e Cesium sbagliati.
4. Mesh OSM: stesso DEM e stesso `--geoid-correction egm96` in `fetch_osm_buildings_bbox.py`.
5. Bbox città storica usata in passato: `S/W/N/E = 44.385 / 8.880 / 44.445 / 8.980`.

**Output:** CSV labels + **viewer Cesium** con raggi LOS (verde) / NLOS (rosso), come i test HK (`build_stadium_visibility_viz.py` → `build_3d_visualization.py`).

**Runtime:** GPU (T4) + build CUDA `_bvh`.  
**Repo:** `https://github.com/andrea444460/gnss_gpu_los_nlos` · branch `feature/cuda-preprocess-area-map`

In [ ]:
# 1) Verifica GPU
!nvidia-smi

In [ ]:
# 2) Clone / aggiorna repo
import os, shutil, subprocess

REPO = "/content/gnss_gpu"
BRANCH = "feature/cuda-preprocess-area-map"
REPO_URL = "https://github.com/andrea444460/gnss_gpu_los_nlos.git"
SCRIPT = f"{REPO}/experiments/label_los_nlos_coords.py"

def _has_script():
    return os.path.isfile(SCRIPT)

if os.path.isdir(REPO) and not _has_script():
    print("Cartella esiste ma manca lo script → rimuovo clone errato")
    shutil.rmtree(REPO)

if not os.path.isdir(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
else:
    subprocess.run(["git", "-C", REPO, "fetch", REPO_URL, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", REPO_URL, BRANCH], check=True)

os.chdir(REPO)
assert _has_script(), f"Manca {SCRIPT} — pusha lo script sul branch prima"
print("OK:", SCRIPT)
!git -C {REPO} rev-parse --short HEAD

In [ ]:
# 3) Dipendenze + build CUDA `_bvh`
# Senza pybind11, CMake salta TUTTI i moduli Python → ModuleNotFoundError.
import os, glob, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/content/gnss_gpu")
os.chdir(REPO)

!pip install -q numpy matplotlib requests scipy rasterio pyproj pybind11
!apt-get -qq install -y cmake

# Arch CUDA: T4=75, L4/A100=80/89 — rileva da nvidia-smi se possibile
arch = "75"
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
        text=True,
    ).strip().splitlines()[0]
    major, minor = out.split(".")
    arch = f"{major}{minor}"
except Exception as exc:
    print("nvidia-smi arch fallback sm_75:", exc)
print("CUDA arch:", arch)

pybind_dir = subprocess.check_output(
    [sys.executable, "-c", "import pybind11; print(pybind11.get_cmake_dir())"],
    text=True,
).strip()
print("pybind11 cmake:", pybind_dir)

build = REPO / "build"
# Se un run precedente non aveva pybind11, la cache cmake e' stale
if (build / "CMakeCache.txt").exists():
    cache = (build / "CMakeCache.txt").read_text(errors="ignore")
    bad = ("pybind11_DIR:PATH=" not in cache) or ("NOTFOUND" in cache and "pybind11" in cache.lower())
    if bad:
        print("CMake cache senza pybind11 -> rebuild pulito")
        shutil.rmtree(build)
build.mkdir(exist_ok=True)
os.chdir(build)
!cmake .. -DCMAKE_CUDA_ARCHITECTURES={arch} -Dpybind11_DIR={pybind_dir}
# Solo BVH (piu' veloce; basta per LOS/NLOS)
!make -j$(nproc) _bvh
os.chdir(REPO)

# Copia .so in python/gnss_gpu/ (README: required for `import gnss_gpu._bvh`)
pkg = REPO / "python" / "gnss_gpu"
sos = list(build.glob("_bvh*.so")) + list(build.glob("_bvh*.pyd"))
if not sos:
    sos = list(build.rglob("_bvh*.so")) + list(build.rglob("_bvh*.pyd"))
print("built artifacts:", [str(p.relative_to(REPO)) for p in sos])
for src in sos:
    dst = pkg / src.name
    shutil.copy2(src, dst)
    print("copied →", dst)

if not sos:
    raise RuntimeError(
        "Nessun _bvh*.so in build/. Controlla che cmake abbia trovato pybind11 "
        "(cerca 'pybind11' nel log cmake) e che make _bvh non sia fallito."
    )

In [ ]:
# 4) Test import BVH
import os, sys, glob
from pathlib import Path

REPO = Path("/content/gnss_gpu")
os.chdir(REPO)
pkg = REPO / "python" / "gnss_gpu"

# Se manca ancora lo .so (build vecchia senza pybind11), ricopia
sos = list((REPO / "build").rglob("_bvh*.so"))
for src in sos:
    dst = pkg / src.name
    if not dst.exists() or dst.stat().st_mtime < src.stat().st_mtime:
        import shutil
        shutil.copy2(src, dst)
        print("copied", src.name, "→", dst)

print("in package:", sorted(p.name for p in pkg.glob("_bvh*")))

os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
# Preferisci il package python/ (dove stanno gli .so copiati)
sys.path = [str(REPO / "python")] + [p for p in sys.path if p not in (str(REPO / "python"), str(REPO / "build"))]

import importlib
import gnss_gpu
importlib.reload(gnss_gpu)
from gnss_gpu import _bvh
print("_bvh OK:", _bvh)
from gnss_gpu.bvh import BVHAccelerator
print("BVHAccelerator OK")

## 5) Dati

In `/content/los_work/data/`:
- `points.csv` / `points.json`
- NAV `*.rnx`
- mesh `triangles.npy` (o genera OSM sotto)

CSV: `id,lat,lon` (con `--rx-alt-mode dem` l’alt CSV viene ignorata).

In [ ]:
# 5a) Cartelle + esempio Genova centro
from pathlib import Path

WORK = Path("/content/los_work")
DATA = WORK / "data"
OUT = WORK / "results"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

example = DATA / "points.csv"
repo_sample = Path("/content/gnss_gpu/experiments/data/genova/2026-10-09_points_genua_n5.csv")
if not example.exists():
    if repo_sample.exists():
        import shutil
        shutil.copy2(repo_sample, example)
        print("Copiato sample Genua n5:", example)
    else:
        example.write_text(
            "id,lat,lon\n"
            "centro,44.4070,8.9330\n"
            "porto,44.4055,8.9280\n",
            encoding="utf-8",
        )
        print("Creato esempio Genova:", example)
else:
    print("Esiste già:", example)

print("File in data/:", sorted(p.name for p in DATA.iterdir()))

In [ ]:
# 5b) Upload (points / NAV / triangles opzionale)
from google.colab import files
from pathlib import Path
import shutil

DATA = Path("/content/los_work/data")
DATA.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    dest = DATA / name
    shutil.move(name, dest)
    print("→", dest)

In [ ]:
# 5c) Mesh OSM Genova + DEM (stesso DEM per footprint edifici)
import os, csv, math
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

coords = DATA / "points.csv"
tri_out = OUT / "genova_osm_triangles.npy"
dem_out = OUT / "genova_area_dem.tif"
BUFFER_DEG = 0.02  # ~2 km intorno ai punti

lats, lons = [], []
with open(coords, newline="", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        lats.append(float(row.get("lat") or row.get("lat_deg")))
        lons.append(float(row.get("lon") or row.get("lon_deg")))

south = min(lats) - BUFFER_DEG
north = max(lats) + BUFFER_DEG
west = min(lons) - BUFFER_DEG
east = max(lons) + BUFFER_DEG
# oppure bbox città completa usata in passato:
# south, west, north, east = 44.3850, 8.8800, 44.4450, 8.9800
print(f"OSM/DEM bbox: S={south:.5f} W={west:.5f} N={north:.5f} E={east:.5f}")

# DEM prima (serve anche ai footprint OSM)
from gnss_gpu.io.dem_download import download_dem_for_bbox
dem_path, ntiles = download_dem_for_bbox(
    south=south - 0.15, west=west - 0.15,  # padding ~15–20 km per horizon colline
    north=north + 0.15, east=east + 0.15,
    output_tif=dem_out, zoom=12,
)
print("DEM:", dem_path, "tiles=", ntiles)

!python {REPO}/experiments/fetch_osm_buildings_bbox.py \
  --south {south} --west {west} --north {north} --east {east} \
  --cache-json {OUT}/genova_osm_cache.json \
  --out-triangles {tri_out} \
  --dem-path {dem_out} \
  --geoid-correction egm96

print("Mesh:", tri_out, "exists=", tri_out.exists())

## 6) Label LOS/NLOS (Genova: DEM on)

`is_los` = edifici AND mask elev AND non bloccato da terreno  
`is_los_buildings` = solo edifici (ablation)

In [ ]:
# 6) Run labeling con DEM (consigliato Genova)
import os
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"

COORDS = DATA / "points.csv"
NAV = next(DATA.glob("*MN.rnx"), None) or next(DATA.glob("*.rnx"), None)
TRIANGLES = OUT / "genova_osm_triangles.npy"
if not TRIANGLES.exists():
    TRIANGLES = next(DATA.glob("*triangles*.npy"), None)
DEM = OUT / "genova_area_dem.tif"

# Tempo allineato al BRDC (Genua n5 = 2026-10-09). Preferisci --utc allo week hardcoded.
UTC = "2026-10-09T12:00:00Z"  # <-- imposta (stesso giorno del NAV)
ELEV_MASK = 10.0

assert COORDS.exists(), COORDS
assert NAV and Path(NAV).exists(), f"Manca NAV in {DATA}"
assert TRIANGLES and Path(TRIANGLES).exists(), "Manca mesh — lancia cella 5c"

OUT_CSV = OUT / "los_nlos_labels.csv"
cmd = f"""python experiments/label_los_nlos_coords.py \
  --coords {COORDS} \
  --nav {NAV} \
  --triangles {TRIANGLES} \
  --utc {UTC} \
  --rx-alt-mode dem \
  --rx-ant-height-m 1.5 \
  --geoid-correction egm96 \
  --elevation-mask-deg {ELEV_MASK} \
  --output-csv {OUT_CSV} \
  --summary-csv {OUT}/los_nlos_summary.csv"""

if DEM.exists():
    cmd += f" --dem-path {DEM}"
else:
    cmd += f" --dem-auto-download --dem-auto-out {OUT}/genova_area_dem.tif"

print(cmd)
get_ipython().system(cmd)

In [ ]:
# 7) Anteprima
import pandas as pd
from pathlib import Path

OUT = Path("/content/los_work/results")
labels = pd.read_csv(OUT / "los_nlos_labels.csv")
summary = pd.read_csv(OUT / "los_nlos_summary.csv")
display(summary)
display(labels.head(20))
print(labels.groupby(["point_id", "is_los"]).size())
if "terrain_blocked_visible" in labels.columns:
    print("terrain_blocked_visible per punto:")
    print(labels.groupby("point_id")["terrain_blocked_visible"].sum())

## 8) Viewer 3D raggi LOS/NLOS (come HK)

Stesso stack dei test Hong Kong / stadio:
`build_stadium_visibility_viz.py` → `build_3d_visualization.py` → HTML Cesium.

Usa le quote DEM dal summary CSV (allineate al labeling). Serve un token [Cesium ion](https://ion.cesium.com/) per il terreno di sfondo.

In [ ]:
# 8a) Token Cesium ion (terreno di sfondo)
import os
os.environ["CESIUM_ION_TOKEN"] = "PASTE_YOUR_CESIUM_ION_TOKEN_HERE"
print("CESIUM_ION_TOKEN set:", bool(os.environ.get("CESIUM_ION_TOKEN", "").strip() and "PASTE" not in os.environ["CESIUM_ION_TOKEN"]))

In [ ]:
# 8b) Genera HTML Cesium con raggi (verde=LOS, rosso=NLOS)
# Riusa build_stadium_visibility_viz.py come per HK / Luigi Ferraris.
import os, csv
from pathlib import Path

REPO = "/content/gnss_gpu"
DATA = Path("/content/los_work/data")
OUT = Path("/content/los_work/results")
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/python:{REPO}/build"
os.environ["MPLBACKEND"] = "Agg"

NAV = next(DATA.glob("*MN.rnx"), None) or next(DATA.glob("*.rnx"), None)
TRIANGLES = OUT / "genova_osm_triangles.npy"
if not TRIANGLES.exists():
    TRIANGLES = next(DATA.glob("*triangles*.npy"), None)
SUMMARY = OUT / "los_nlos_summary.csv"
assert SUMMARY.exists(), "Lancia prima il labeling (cella 6)"
assert NAV and Path(NAV).exists()
assert TRIANGLES and Path(TRIANGLES).exists()

# Lat/lon + alt DEM dal summary (stesse quote del labeling)
viz_csv = OUT / "viz_points_dem_alt.csv"
with open(SUMMARY, newline="", encoding="utf-8") as fh:
    rows = list(csv.DictReader(fh))
with open(viz_csv, "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["id", "lat", "lon", "alt"])
    w.writeheader()
    for r in rows:
        w.writerow({
            "id": r["point_id"],
            "lat": r["lat_deg"],
            "lon": r["lon_deg"],
            "alt": r["alt_m"],
        })
print("viz coords:", viz_csv, "n=", len(rows))

OUT_HTML = OUT / "genova_los_rays_viz.html"
REF_CSV = OUT / "genova_los_rays_viz_reference.csv"
n_pts = len(rows)
# Usa --start-utc (evita week/tow sbagliati). Allinea al giorno BRDC.
UTC = "2026-10-09T12:00:00Z"

token = os.environ.get("CESIUM_ION_TOKEN", "").strip()
cmd = f"""python experiments/build_stadium_visibility_viz.py \
  --latlon-csv {viz_csv} \
  --triangles-npy {TRIANGLES} \
  --nav {NAV} \
  --out-html {OUT_HTML} \
  --reference-out {REF_CSV} \
  --area-name GenovaCoords \
  --start-utc {UTC} \
  --dt-s 0 \
  --n-epochs {n_pts} \
  --traj-step 1 \
  --elevation-mask-deg 10 \
  --export-mesh-glb \
  --plateau-glb-radius-m 2500 \
  --plateau-glb-max-tris 800000"""
if token and "PASTE" not in token:
    cmd += f" --cesium-ion-token {token}"

print(cmd)
get_ipython().system(cmd)
print("HTML:", OUT_HTML, "exists=", OUT_HTML.exists())

In [ ]:
# 8c) Apri viewer nel browser (IFrame) — serve la cartella results via HTTP
from pathlib import Path
from IPython.display import IFrame, display
import subprocess, time

OUT = Path("/content/los_work/results")
html = OUT / "genova_los_rays_viz.html"
if not html.exists():
    raise FileNotFoundError(html)

# Ponytail: one local server for Cesium assets (GLB + HTML)
subprocess.Popen(
    f"python -m http.server 8765 --directory {OUT}",
    shell=True,
)
time.sleep(1)
display(IFrame(src=f"http://localhost:8765/{html.name}", width="100%", height=700))
print("Se l'IFrame e' vuoto, apri in nuova scheda:", f"http://localhost:8765/{html.name}")

In [ ]:
# 9) Download CSV + HTML (+ GLB mesh se presente)
from google.colab import files
from pathlib import Path

OUT = Path("/content/los_work/results")
for name in (
    "los_nlos_labels.csv",
    "los_nlos_summary.csv",
    "genova_los_rays_viz.html",
    "genova_los_rays_viz_mesh.glb",
):
    p = OUT / name
    if p.exists():
        files.download(str(p))
        print("↓", p.name)
    else:
        print("skip (manca):", name)